# Arena Ranker — Kaggle GPU 训练与推理

本 notebook 在 Kaggle 提供的 **GPU 环境（T4 x2 / P100 x1）** 中完成：

1. 安装依赖
2. 设置项目源码
3. 使用 **QLoRA** 微调 `Qwen/Qwen3-0.6B` 偏好分类模型
4. 推理并生成 `submission.csv`

### 核心架构

```
prompt + response_a + response_b
        ↓
  apply_chat_template (system + user)
        ↓
  Qwen3-0.6B (4-bit NF4 量化)
        ↓
  AutoModelForSequenceClassification
        ↓
  score (分类头, 3-class logits)
```

> **前置准备**
>
> | 项目 | 操作 |
> |------|------|
> | **竞赛数据** | Notebook 右侧 → Add Input → 搜索并添加竞赛数据集 |
> | **GPU** | Settings → Accelerator → 选择 **GPU T4 ×2** 或 **GPU P100** |
> | **联网** | Settings → Internet → **On**（用于下载 HuggingFace 模型）|
>
> 如果不想联网下载模型，请参考最后一节「离线模式」。


## 1. 安装依赖

Kaggle 环境已预装 PyTorch，这里补装 QLoRA 训练所需的包。

In [ ]:
!pip install -q \
    "transformers>=4.55.0" \
    "peft>=0.17.0" \
    "bitsandbytes>=0.45.0" \
    "datasets>=3.0.0" \
    "accelerate>=1.0.0" \
    "scikit-learn>=1.5.0" \
    "tqdm>=4.66.0" \
    "pyyaml>=6.0.2"

## 2. 写入项目源码

将 `arena_ranker` 包的所有源文件写入 `/kaggle/working/arena_ranker/`。

> **替代方案**：也可以把本仓库上传为 Kaggle Dataset，
> 然后 `!pip install /kaggle/input/<your-dataset-slug>/` 来安装。

In [ ]:
import base64
from pathlib import Path

PKG_DIR = Path("/kaggle/working/arena_ranker")
PKG_DIR.mkdir(parents=True, exist_ok=True)

_FILES = {
    "__init__.py": "IiIiQXJlbmEgUmFua2VyIOKAlCBRTG9SQSDlvq7osIMgUXdlbjMtMC42QiDnlKjkuo4gQ2hhdEJvdCBBcmVuYSDlgY/lpb3pooTmtYvjgIIiIiIK",
    "config.py": "IiIiCumFjee9ruaooeWdlyDigJQg5a6a5LmJ6K6t57uD5ZKM5o6o55CG5omA6ZyA55qE5YWo6YOo6buY6K6k5Y+C5pWw44CCCgrmnKzmqKHlnZfph4fnlKggZGF0YWNsYXNzIOe7hOe7h+mFjee9ru+8jOaUr+aMgSBZQU1MIOaMgeS5heWMluOAggrkuLvopoHliIbkuLrkuInpg6jliIbvvJoKICAtIERhdGFDb25maWc6ICDmlbDmja7ot6/lvoTjgIHmlofmnKzmiKrmlq3jgIHpqozor4Hpm4bmr5TkvosKICAtIE1vZGVsQ29uZmlnOiDln7rluqfmqKHlnovjgIHph4/ljJbjgIFMb1JBIOWPguaVsAogIC0gVHJhaW5pbmdDb25maWc6IOWtpuS5oOeOh+OAgWJhdGNoIHNpemXjgIFlcG9jaCDnrYkgVHJhaW5lciDlj4LmlbAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB5YW1sCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5qCH562+55u45YWz5bi46YePCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkxBQkVMX0NPTFVNTlMgPSBbIndpbm5lcl9tb2RlbF9hIiwgIndpbm5lcl9tb2RlbF9iIiwgIndpbm5lcl90aWUiXQpMQUJFTF9UT19JRCA9IHsid2lubmVyX21vZGVsX2EiOiAwLCAid2lubmVyX21vZGVsX2IiOiAxLCAid2lubmVyX3RpZSI6IDJ9CklEX1RPX0xBQkVMID0ge3Y6IGsgZm9yIGssIHYgaW4gTEFCRUxfVE9fSUQuaXRlbXMoKX0KTlVNX0xBQkVMUyA9IDMKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOWvueivneaooeadv+W4uOmHjwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpTWVNURU1fUFJPTVBUID0gIiIiUGxlYXNlIGFjdCBhcyBhbiBpbXBhcnRpYWwganVkZ2UgYW5kIGV2YWx1YXRlIHRoZSBxdWFsaXR5IG9mIHRoZSByZXNwb25zZXMgcHJvdmlkZWQgYnkgdHdvCkFJIGFzc2lzdGFudHMgdG8gdGhlIHVzZXIgcXVlc3Rpb24gZGlzcGxheWVkIGJlbG93LiBZb3Ugc2hvdWxkIGNob29zZSB0aGUgYXNzaXN0YW50IHRoYXQKZm9sbG93cyB0aGUgdXNlcuKAmXMgaW5zdHJ1Y3Rpb25zIGFuZCBhbnN3ZXJzIHRoZSB1c2Vy4oCZcyBxdWVzdGlvbiBiZXR0ZXIuIFlvdXIgZXZhbHVhdGlvbgpzaG91bGQgY29uc2lkZXIgZmFjdG9ycyBzdWNoIGFzIHRoZSBoZWxwZnVsbmVzcywgcmVsZXZhbmNlLCBhY2N1cmFjeSwgZGVwdGgsIGNyZWF0aXZpdHksCmFuZCBsZXZlbCBvZiBkZXRhaWwgb2YgdGhlaXIgcmVzcG9uc2VzLiBCZWdpbiB5b3VyIGV2YWx1YXRpb24gYnkgY29tcGFyaW5nIHRoZSB0d28KcmVzcG9uc2VzIGFuZCBwcm92aWRlIGEgc2hvcnQgZXhwbGFuYXRpb24uIEF2b2lkIGFueSBwb3NpdGlvbiBiaWFzZXMgYW5kIGVuc3VyZSB0aGF0IHRoZQpvcmRlciBpbiB3aGljaCB0aGUgcmVzcG9uc2VzIHdlcmUgcHJlc2VudGVkIGRvZXMgbm90IGluZmx1ZW5jZSB5b3VyIGRlY2lzaW9uLiBEbyBub3QgYWxsb3cKdGhlIGxlbmd0aCBvZiB0aGUgcmVzcG9uc2VzIHRvIGluZmx1ZW5jZSB5b3VyIGV2YWx1YXRpb24uIERvIG5vdCBmYXZvciBjZXJ0YWluIG5hbWVzIG9mCnRoZSBhc3Npc3RhbnRzLiBCZSBhcyBvYmplY3RpdmUgYXMgcG9zc2libGUuIEFmdGVyIHByb3ZpZGluZyB5b3VyIGV4cGxhbmF0aW9uLCBvdXRwdXQgeW91cgpmaW5hbCB2ZXJkaWN0IGJ5IHN0cmljdGx5IGZvbGxvd2luZyB0aGlzIGZvcm1hdDogIltbQV1dIiBpZiBhc3Npc3RhbnQgQSBpcyBiZXR0ZXIsICJbW0JdXSIKaWYgYXNzaXN0YW50IEIgaXMgYmV0dGVyLCBhbmQgIltbQ11dIiBmb3IgYSB0aWUuIiIiCgpVU0VSX1RFTVBMQVRFID0gIntjb252ZXJzYXRpb259IgpWRVJESUNUX1NVRkZJWCA9ICJ2ZXJkaWN0IGlzOiBbWyIKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBEYXRhQ29uZmlnCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgRGF0YUNvbmZpZzoKICAgIHRyYWluX3BhdGg6IHN0ciA9ICJ0cmFpbi5jc3YiCiAgICB0ZXN0X3BhdGg6IHN0ciA9ICJ0ZXN0LmNzdiIKICAgIHRleHRfbWF4X2NoYXJzOiBpbnQgPSA2MDAwCiAgICB2YWxpZGF0aW9uX3NpemU6IGZsb2F0ID0gMC4xCiAgICByYW5kb21fc3RhdGU6IGludCA9IDQyCiAgICBpbmNsdWRlX3N3YXBfdHJhaW46IGJvb2wgPSBUcnVlCiAgICBpbmNsdWRlX3N3YXBfdHRhOiBib29sID0gVHJ1ZQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIE1vZGVsQ29uZmlnCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgTW9kZWxDb25maWc6CiAgICBtb2RlbF9uYW1lOiBzdHIgPSAiUXdlbi9Rd2VuMy0wLjZCIgogICAgY2FjaGVfZGlyOiBzdHIgfCBOb25lID0gTm9uZQogICAgbG9jYWxfZmlsZXNfb25seTogYm9vbCA9IEZhbHNlCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAxMDI0CgogICAgIyAtLS0gNC1iaXQg6YeP5YyWIChRTG9SQSkgLS0tCiAgICBsb2FkX2luXzRiaXQ6IGJvb2wgPSBUcnVlCiAgICBibmJfNGJpdF9xdWFudF90eXBlOiBzdHIgPSAibmY0IgogICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudDogYm9vbCA9IFRydWUKCiAgICAjIC0tLSBMb1JBIC0tLQogICAgdXNlX2xvcmE6IGJvb2wgPSBUcnVlCiAgICBsb3JhX3I6IGludCA9IDMyCiAgICBsb3JhX2FscGhhOiBpbnQgPSA2NAogICAgbG9yYV9kcm9wb3V0OiBmbG9hdCA9IDAuMDUKICAgIGxvcmFfYmlhczogc3RyID0gIm5vbmUiCiAgICBsb3JhX3RhcmdldF9tb2R1bGVzOiBsaXN0W3N0cl0gPSBmaWVsZCgKICAgICAgICBkZWZhdWx0X2ZhY3Rvcnk9bGFtYmRhOiBbCiAgICAgICAgICAgICJxX3Byb2oiLCAia19wcm9qIiwgInZfcHJvaiIsICJvX3Byb2oiLAogICAgICAgICAgICAiZ2F0ZV9wcm9qIiwgInVwX3Byb2oiLCAiZG93bl9wcm9qIiwKICAgICAgICBdCiAgICApCiAgICBsb3JhX21vZHVsZXNfdG9fc2F2ZTogbGlzdFtzdHJdID0gZmllbGQoCiAgICAgICAgZGVmYXVsdF9mYWN0b3J5PWxhbWJkYTogWyJzY29yZSJdCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAgVHJhaW5pbmdDb25maWcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBUcmFpbmluZ0NvbmZpZzoKICAgIG91dHB1dF9kaXI6IHN0ciA9ICJhcnRpZmFjdHMvZGVmYXVsdCIKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0ID0gMmUtNAogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMDEKICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZTogaW50ID0gMgogICAgcGVyX2RldmljZV9ldmFsX2JhdGNoX3NpemU6IGludCA9IDQKICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwczogaW50ID0gOAogICAgbnVtX3RyYWluX2Vwb2NoczogaW50ID0gMwogICAgd2FybXVwX3JhdGlvOiBmbG9hdCA9IDAuMQogICAgd2FybXVwX3N0ZXBzOiBpbnQgPSAwCiAgICBscl9zY2hlZHVsZXJfdHlwZTogc3RyID0gImNvc2luZSIKICAgIG9wdGltOiBzdHIgPSAicGFnZWRfYWRhbXdfMzJiaXQiCiAgICBmcDE2OiBib29sID0gVHJ1ZQogICAgYmYxNjogYm9vbCA9IEZhbHNlCiAgICBkZHBfZmluZF91bnVzZWRfcGFyYW1ldGVyczogYm9vbCA9IEZhbHNlCiAgICBncmFkaWVudF9jaGVja3BvaW50aW5nOiBib29sID0gVHJ1ZQogICAgbG9nZ2luZ19zdGVwczogaW50ID0gNTAKICAgIGV2YWxfc3RyYXRlZ3k6IHN0ciA9ICJlcG9jaCIKICAgIHNhdmVfc3RyYXRlZ3k6IHN0ciA9ICJlcG9jaCIKICAgIHNhdmVfdG90YWxfbGltaXQ6IGludCA9IDIKICAgIGxvYWRfYmVzdF9tb2RlbF9hdF9lbmQ6IGJvb2wgPSBUcnVlCiAgICBtZXRyaWNfZm9yX2Jlc3RfbW9kZWw6IHN0ciA9ICJsb2dfbG9zcyIKICAgIGdyZWF0ZXJfaXNfYmV0dGVyOiBib29sID0gRmFsc2UKICAgIHNlZWQ6IGludCA9IDQyCiAgICByZXBvcnRfdG86IHN0ciA9ICJub25lIgogICAgZGF0YWxvYWRlcl9udW1fd29ya2VyczogaW50ID0gMAoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIEFwcENvbmZpZyAo6aG25bGC6IGa5ZCIKQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIEFwcENvbmZpZzoKICAgIGRhdGE6IERhdGFDb25maWcgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9RGF0YUNvbmZpZykKICAgIG1vZGVsOiBNb2RlbENvbmZpZyA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1Nb2RlbENvbmZpZykKICAgIHRyYWluaW5nOiBUcmFpbmluZ0NvbmZpZyA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1UcmFpbmluZ0NvbmZpZykKCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4gYXNkaWN0KHNlbGYpCgogICAgZGVmIHNhdmUoc2VsZiwgcGF0aDogc3RyIHwgUGF0aCkgLT4gTm9uZToKICAgICAgICBvdXRwdXRfcGF0aCA9IFBhdGgocGF0aCkKICAgICAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIG91dHB1dF9wYXRoLndyaXRlX3RleHQoCiAgICAgICAgICAgIHlhbWwuc2FmZV9kdW1wKHNlbGYudG9fZGljdCgpLCBzb3J0X2tleXM9RmFsc2UpLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKCgpkZWYgbG9hZF9jb25maWcocGF0aDogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBBcHBDb25maWc6CiAgICAiIiLku44gWUFNTCDmlofku7bliqDovb3phY3nva7vvJtwYXRoPU5vbmUg5pe26L+U5Zue5YWo6YOo6buY6K6k5YC844CCIiIiCiAgICBpZiBwYXRoIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIEFwcENvbmZpZygpCiAgICByYXcgPSB5YW1sLnNhZmVfbG9hZChQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHRyYWluaW5nX3JhdyA9IHJhdy5nZXQoInRyYWluaW5nIiwge30pCgogICAgIyDlhbzlrrnml6fphY3nva46IOabvuivr+aKiiAwLjEg6L+Z57G75q+U5L6L5YaZ6L+bIHdhcm11cF9zdGVwc+OAggogICAgbGVnYWN5X3dhcm11cCA9IHRyYWluaW5nX3Jhdy5nZXQoIndhcm11cF9zdGVwcyIpCiAgICBpZiAid2FybXVwX3JhdGlvIiBub3QgaW4gdHJhaW5pbmdfcmF3IGFuZCBpc2luc3RhbmNlKGxlZ2FjeV93YXJtdXAsIGZsb2F0KToKICAgICAgICBpZiAwLjAgPD0gbGVnYWN5X3dhcm11cCA8PSAxLjA6CiAgICAgICAgICAgIHRyYWluaW5nX3Jhd1sid2FybXVwX3JhdGlvIl0gPSBsZWdhY3lfd2FybXVwCiAgICAgICAgICAgIHRyYWluaW5nX3Jhd1sid2FybXVwX3N0ZXBzIl0gPSAwCgogICAgcmV0dXJuIEFwcENvbmZpZygKICAgICAgICBkYXRhPURhdGFDb25maWcoKipyYXcuZ2V0KCJkYXRhIiwge30pKSwKICAgICAgICBtb2RlbD1Nb2RlbENvbmZpZygqKnJhdy5nZXQoIm1vZGVsIiwge30pKSwKICAgICAgICB0cmFpbmluZz1UcmFpbmluZ0NvbmZpZygqKnRyYWluaW5nX3JhdyksCiAgICApCg==",
    "data.py": "IiIiCuaVsOaNruWkhOeQhuaooeWdlyDigJQg5Yqg6L29IENTVuOAgeaWh+acrOa4hea0l+OAgWNoYXQgdGVtcGxhdGUgdG9rZW5pemF0aW9u44CBbWV0cmljcyDorqHnrpfjgIIKCuaguOW/g+a1geeoi++8mgogIDEuIOivu+WPliBDU1Yg4oaSIOa4hea0l+aWh+acrOWtl+autSAo6Kej5p6QIEpTT04g5pWw57uEIC8gUHl0aG9uIGxpc3Qg562J5qC85byPKQogIDIuIOS9v+eUqCB0b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgpIOaehOW7uiBzeXN0ZW0gKyB1c2VyIOWvueivnQogIDMuIOmAmui/hyBEYXRhc2V0Lm1hcCgpIOWujOaIkCB0b2tlbml6YXRpb27vvIzovpPlh7ogaW5wdXRfaWRzIC8gYXR0ZW50aW9uX21hc2sgLyBsYWJlbHMKICA0LiDmj5DkvpsgY29tcHV0ZV9tZXRyaWNzKCkg5L6bIFRyYWluZXIg5L2/55SoCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQganNvbgpmcm9tIGl0ZXJ0b29scyBpbXBvcnQgemlwX2xvbmdlc3QKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBkYXRhc2V0cyBpbXBvcnQgRGF0YXNldApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmUsIGxvZ19sb3NzCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKCmZyb20gYXJlbmFfcmFua2VyLmNvbmZpZyBpbXBvcnQgKAogICAgTEFCRUxfQ09MVU1OUywKICAgIExBQkVMX1RPX0lELAogICAgU1lTVEVNX1BST01QVCwKICAgIFVTRVJfVEVNUExBVEUsCiAgICBWRVJESUNUX1NVRkZJWCwKICAgIERhdGFDb25maWcsCikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDmlofmnKzop6PmnpAgJiDmuIXmtJcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlOiBBbnkpIC0+IGxpc3Rbc3RyXToKICAgICIiIgogICAg5bCG5Y6f5aeLIENTViDlrZfmrrXop6PmnpDkuLrlrZfnrKbkuLLliJfooajjgIIKICAgIOaUr+aMgeS4ieenjei+k+WFpeagvOW8j++8mgogICAgICAtIOaZrumAmuWtl+espuS4siDihpIg55u05o6l5YyF5YWl5YiX6KGoCiAgICAgIC0gSlNPTiDmlbDnu4TlrZfnrKbkuLIg4oaSIGpzb24ubG9hZHMg6Kej5p6QCiAgICAgIC0gUHl0aG9uIGxpc3Qg5a2X56ym5LiyIOKGkiBhc3QubGl0ZXJhbF9ldmFsIOino+aekAogICAgIiIiCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KToKICAgICAgICByZXR1cm4gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiB2YWx1ZV0KICAgIGlmIHZhbHVlIGlzIE5vbmUgb3IgKGlzaW5zdGFuY2UodmFsdWUsIGZsb2F0KSBhbmQgcGQuaXNuYSh2YWx1ZSkpOgogICAgICAgIHJldHVybiBbXQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CiAgICAgICAgcmV0dXJuIFtzdHIodmFsdWUpXQogICAgdGV4dCA9IHZhbHVlLnN0cmlwKCkKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiBbXQogICAgZm9yIHBhcnNlciBpbiAoanNvbi5sb2FkcywgYXN0LmxpdGVyYWxfZXZhbCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBwYXJzZXIodGV4dCkKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwYXJzZWQsIGxpc3QpOgogICAgICAgICAgICAgICAgcmV0dXJuIFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gcGFyc2VkXQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgU3ludGF4RXJyb3IpOgogICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIFt0ZXh0XQoKCmRlZiBfbm9ybWFsaXplX2NvbnZlcnNhdGlvbl90dXJucyh2YWx1ZTogQW55LCBtYXhfY2hhcnM6IGludCkgLT4gbGlzdFtzdHJdOgogICAgIiIi5bCG5Y6f5aeL5a2X5q616L2s5Li65oyJ6L2u5qyh5L+d5bqP55qE5a2X56ym5Liy5YiX6KGo77yM5bm25Zyo5oC75a2X56ym6aKE566X5YaF5oiq5pat44CCIiIiCiAgICBjaHVua3MgPSBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlKQogICAgbm9ybWFsaXplZCA9IFtjaHVuay5zdHJpcCgpIGZvciBjaHVuayBpbiBjaHVua3MgaWYgc3RyKGNodW5rKS5zdHJpcCgpXQogICAgaWYgbm90IG5vcm1hbGl6ZWQ6CiAgICAgICAgcmV0dXJuIFtdCgogICAga2VwdDogbGlzdFtzdHJdID0gW10KICAgIHVzZWRfY2hhcnMgPSAwCiAgICBmb3IgY2h1bmsgaW4gbm9ybWFsaXplZDoKICAgICAgICByZW1haW5pbmcgPSBtYXhfY2hhcnMgLSB1c2VkX2NoYXJzCiAgICAgICAgaWYgcmVtYWluaW5nIDw9IDA6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgdHJ1bmNhdGVkID0gY2h1bmtbOnJlbWFpbmluZ10KICAgICAgICBpZiB0cnVuY2F0ZWQ6CiAgICAgICAgICAgIGtlcHQuYXBwZW5kKHRydW5jYXRlZCkKICAgICAgICAgICAgdXNlZF9jaGFycyArPSBsZW4odHJ1bmNhdGVkKQogICAgcmV0dXJuIGtlcHQKCgpkZWYgX2J1aWxkX2NvbnZlcnNhdGlvbl90ZXh0KAogICAgcHJvbXB0X3R1cm5zOiBsaXN0W3N0cl0sCiAgICByZXNwb25zZV9hX3R1cm5zOiBsaXN0W3N0cl0sCiAgICByZXNwb25zZV9iX3R1cm5zOiBsaXN0W3N0cl0sCikgLT4gc3RyOgogICAgIiIi5oyJ6L2u5qyh5Lqk6ZSZ5ou85o6lIHByb21wdCAvIEEgLyBC77yM5L+d55WZ5aSa6L2u5a+56K+d57uT5p6E44CCIiIiCiAgICBoZWFkID0gIjx8VGhlIFN0YXJ0IG9mIENvbnZlcnNhdGlvbiBiZXR3ZWVuIGEgVXNlciBhbmQgdHdvIEFzc2lzdGFudHN8PiIKICAgIHRhaWwgPSAiPHxUaGUgRW5kIG9mIENvbnZlcnNhdGlvbiBiZXR3ZWVuIGEgVXNlciBhbmQgdHdvIEFzc2lzdGFudHN8PlxuIgogICAgcGFydHMgPSBbXQogICAgZm9yIHByb21wdCwgcmVzcG9uc2VfYSwgcmVzcG9uc2VfYiBpbiB6aXBfbG9uZ2VzdCgKICAgICAgICBwcm9tcHRfdHVybnMsCiAgICAgICAgcmVzcG9uc2VfYV90dXJucywKICAgICAgICByZXNwb25zZV9iX3R1cm5zLAogICAgICAgIGZpbGx2YWx1ZT0ibnVsbCIsCiAgICApOgogICAgICAgIHBhcnRzLmFwcGVuZCgKICAgICAgICAgICAgZiJcbiMjIyBVc2VyOlxue3Byb21wdH1cblxuIyMjIEFzc2lzdGFudCBBOlxue3Jlc3BvbnNlX2F9XG5cbiMjIyBBc3Npc3RhbnQgQjpcbntyZXNwb25zZV9ifVxuIgogICAgICAgICkKICAgIHJldHVybiBoZWFkICsgIiIuam9pbihwYXJ0cykgKyB0YWlsCgoKZGVmIF9idWlsZF9sYWJlbChyb3c6IHBkLlNlcmllcykgLT4gaW50OgogICAgIiIi5LuOIG9uZS1ob3Qg5qCH562+5YiXIOKGkiDljZXkuIDmlbTmlbDmoIfnrb4gKDA9QeiDnCwgMT1C6IOcLCAyPeW5s+WxgCnjgIIiIiIKICAgIGZvciBjb2wsIGxhYmVsX2lkIGluIExBQkVMX1RPX0lELml0ZW1zKCk6CiAgICAgICAgaWYgaW50KHJvd1tjb2xdKSA9PSAxOgogICAgICAgICAgICByZXR1cm4gbGFiZWxfaWQKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiLml6DmlYjmoIfnrb7ooYw6IHtyb3dbTEFCRUxfQ09MVU1OU10udG9fZGljdCgpfSIpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5pWw5o2u5Yqg6L29ICYg6aKE5aSE55CGCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbG9hZF9hbmRfcHJlcHJvY2VzcygKICAgIGNzdl9wYXRoOiBzdHIsCiAgICBtYXhfY2hhcnM6IGludCA9IDYwMDAsCiAgICBpc190cmFpbjogYm9vbCA9IFRydWUsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiCiAgICDliqDovb0gQ1NWIOW5tumihOWkhOeQhuaWh+acrOWtl+auteOAggoKICAgIFJldHVybnM6CiAgICAgICAgRGF0YUZyYW1l77yM5YyF5ZCrIGlkIC8gcHJvbXB0X3R1cm5zIC8gcmVzcG9uc2VfYV90dXJucyAvIHJlc3BvbnNlX2JfdHVybnMKICAgICAgICDku6Xlj4ogKOS7heiuree7g+mbhikgbGFiZWxzIOWIl+OAggogICAgIiIiCiAgICBkZiA9IHBkLnJlYWRfY3N2KGNzdl9wYXRoKQogICAgZGZbInByb21wdF90dXJucyJdID0gZGZbInByb21wdCJdLm1hcCgKICAgICAgICBsYW1iZGEgeDogX25vcm1hbGl6ZV9jb252ZXJzYXRpb25fdHVybnMoeCwgbWF4X2NoYXJzKQogICAgKQogICAgZGZbInJlc3BvbnNlX2FfdHVybnMiXSA9IGRmWyJyZXNwb25zZV9hIl0ubWFwKAogICAgICAgIGxhbWJkYSB4OiBfbm9ybWFsaXplX2NvbnZlcnNhdGlvbl90dXJucyh4LCBtYXhfY2hhcnMpCiAgICApCiAgICBkZlsicmVzcG9uc2VfYl90dXJucyJdID0gZGZbInJlc3BvbnNlX2IiXS5tYXAoCiAgICAgICAgbGFtYmRhIHg6IF9ub3JtYWxpemVfY29udmVyc2F0aW9uX3R1cm5zKHgsIG1heF9jaGFycykKICAgICkKICAgIGlmIGlzX3RyYWluOgogICAgICAgIGRmWyJsYWJlbHMiXSA9IGRmLmFwcGx5KF9idWlsZF9sYWJlbCwgYXhpcz0xKQogICAgcmV0dXJuIGRmCgoKZGVmIHNwbGl0X3RyYWluX3ZhbGlkKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgIGNvbmZpZzogRGF0YUNvbmZpZywKKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIHBkLkRhdGFGcmFtZV06CiAgICAiIiLmjIkgc3RyYXRpZmllZCBzcGxpdCDliJLliIborq3nu4Ppm4blkozpqozor4Hpm4bjgIIiIiIKICAgIHRyYWluX2RmLCB2YWxpZF9kZiA9IHRyYWluX3Rlc3Rfc3BsaXQoCiAgICAgICAgZGYsCiAgICAgICAgdGVzdF9zaXplPWNvbmZpZy52YWxpZGF0aW9uX3NpemUsCiAgICAgICAgcmFuZG9tX3N0YXRlPWNvbmZpZy5yYW5kb21fc3RhdGUsCiAgICAgICAgc3RyYXRpZnk9ZGZbImxhYmVscyJdLAogICAgKQogICAgcmV0dXJuIHRyYWluX2RmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSksIHZhbGlkX2RmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDaGF0IFRlbXBsYXRlIFRva2VuaXphdGlvbgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIF9idWlsZF9jaGF0X21lc3NhZ2VzKGNvbnZlcnNhdGlvbjogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgIiIiCiAgICDmnoTlu7rlr7nor53mtojmga/liJfooajvvIznlKjkuo4gYXBwbHlfY2hhdF90ZW1wbGF0ZeOAggogICAgICAtIHN5c3RlbTog6K+E5aeU6KeS6Imy5oyH5LukCiAgICAgIC0gdXNlcjogICDkv53nlZnova7mrKHnu5PmnoTnmoTlrozmlbTlr7nor50KICAgICIiIgogICAgcmV0dXJuIFsKICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1fUFJPTVBUfSwKICAgICAgICB7CiAgICAgICAgICAgICJyb2xlIjogInVzZXIiLAogICAgICAgICAgICAiY29udGVudCI6IFVTRVJfVEVNUExBVEUuZm9ybWF0KGNvbnZlcnNhdGlvbj1jb252ZXJzYXRpb24pLAogICAgICAgIH0sCiAgICBdCgoKZGVmIF90b2tlbml6ZV9zaW5nbGUoZXhhbXBsZTogZGljdCwgdG9rZW5pemVyLCBtYXhfbGVuZ3RoOiBpbnQpIC0+IGRpY3Q6CiAgICAiIiIKICAgIOWvueWNleadoeagt+acrOWBmiB0b2tlbml6YXRpb27vvIjkvpsgRGF0YXNldC5tYXAg6LCD55So77yJ44CCCgogICAg5q2l6aqk77yaCiAgICAgIDEuIOeUqCBhcHBseV9jaGF0X3RlbXBsYXRlIOWwhuWvueivneagvOW8j+WMluS4uuaWh+acrAogICAgICAyLiDnlKggdG9rZW5pemVyIOe8lueggeW5tuaIquaWreWIsCBtYXhfbGVuZ3RoCiAgICAiIiIKICAgIGNvbnZlcnNhdGlvbiA9IF9idWlsZF9jb252ZXJzYXRpb25fdGV4dCgKICAgICAgICBleGFtcGxlWyJwcm9tcHRfdHVybnMiXSwKICAgICAgICBleGFtcGxlWyJyZXNwb25zZV9hX3R1cm5zIl0sCiAgICAgICAgZXhhbXBsZVsicmVzcG9uc2VfYl90dXJucyJdLAogICAgKQogICAgbWVzc2FnZXMgPSBfYnVpbGRfY2hhdF9tZXNzYWdlcyhjb252ZXJzYXRpb24pCiAgICAjIOWFiOW+l+WIsOagvOW8j+WMluaWh+acrO+8jOWGjeWNleeLrCB0b2tlbml6ZSDku6Xnsr7noa7mjqfliLbmiKrmlq0KICAgIHRleHQgPSB0b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICBtZXNzYWdlcywKICAgICAgICB0b2tlbml6ZT1GYWxzZSwKICAgICAgICBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwKICAgICkgKyBWRVJESUNUX1NVRkZJWAogICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICB0ZXh0LAogICAgICAgIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlLCAgIyBjaGF0IHRlbXBsYXRlIOW3sue7j+WMheWQq+aJgOacieeJueauiiB0b2tlbgogICAgKQogICAgcmV0dXJuIGVuY29kZWQKCgpkZWYgYnVpbGRfZGF0YXNldCgKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICB0b2tlbml6ZXIsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAxMDI0LAogICAgaXNfdHJhaW46IGJvb2wgPSBUcnVlLAogICAgaW5jbHVkZV9zd2FwOiBib29sID0gRmFsc2UsCiAgICBzd2FwX3BhaXJzOiBib29sID0gRmFsc2UsCikgLT4gRGF0YXNldDoKICAgICIiIgogICAg5LuOIERhdGFGcmFtZSDmnoTlu7ogdG9rZW5pemVkIEh1Z2dpbmdGYWNlIERhdGFzZXTjgIIKCiAgICDovpPlh7rliJfvvIjorq3nu4PvvIk6IGlucHV0X2lkcywgYXR0ZW50aW9uX21hc2ssIGxhYmVscwogICAg6L6T5Ye65YiX77yI5rWL6K+V77yJOiBpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrCiAgICAiIiIKICAgIGNvbHMgPSBbImlkIiwgInByb21wdF90dXJucyIsICJyZXNwb25zZV9hX3R1cm5zIiwgInJlc3BvbnNlX2JfdHVybnMiXQogICAgaWYgaXNfdHJhaW46CiAgICAgICAgY29scy5hcHBlbmQoImxhYmVscyIpCiAgICBiYXNlX2RmID0gZGZbY29sc10uY29weSgpCgogICAgaWYgc3dhcF9wYWlycyBvciBpbmNsdWRlX3N3YXA6CiAgICAgICAgc3dhcF9kZiA9IGJhc2VfZGYuY29weSgpCiAgICAgICAgc3dhcF9kZlsicmVzcG9uc2VfYV90dXJucyJdID0gYmFzZV9kZlsicmVzcG9uc2VfYl90dXJucyJdCiAgICAgICAgc3dhcF9kZlsicmVzcG9uc2VfYl90dXJucyJdID0gYmFzZV9kZlsicmVzcG9uc2VfYV90dXJucyJdCiAgICAgICAgaWYgaXNfdHJhaW46CiAgICAgICAgICAgIHN3YXBfZGZbImxhYmVscyJdID0gc3dhcF9kZlsibGFiZWxzIl0ubWFwKAogICAgICAgICAgICAgICAgbGFtYmRhIHg6IDEgaWYgeCA9PSAwIGVsc2UgMCBpZiB4ID09IDEgZWxzZSB4CiAgICAgICAgICAgICkKICAgICAgICBpZiBzd2FwX3BhaXJzOgogICAgICAgICAgICBiYXNlX2RmID0gc3dhcF9kZgogICAgICAgIGVsaWYgaW5jbHVkZV9zd2FwOgogICAgICAgICAgICBiYXNlX2RmID0gcGQuY29uY2F0KFtiYXNlX2RmLCBzd2FwX2RmXSwgaWdub3JlX2luZGV4PVRydWUpCgogICAgZGF0YXNldCA9IERhdGFzZXQuZnJvbV9wYW5kYXMoYmFzZV9kZiwgcHJlc2VydmVfaW5kZXg9RmFsc2UpCgogICAgZGF0YXNldCA9IGRhdGFzZXQubWFwKAogICAgICAgIGxhbWJkYSB4OiBfdG9rZW5pemVfc2luZ2xlKHgsIHRva2VuaXplciwgbWF4X2xlbmd0aCksCiAgICAgICAgZGVzYz0iVG9rZW5pemluZyIsCiAgICApCgogICAgIyDnp7vpmaTmlofmnKzliJfvvIzlj6rkv53nlZnmqKHlnovpnIDopoHnmoTmlbDlgLzliJcKICAgIHJlbW92ZV9jb2xzID0gWyJwcm9tcHRfdHVybnMiLCAicmVzcG9uc2VfYV90dXJucyIsICJyZXNwb25zZV9iX3R1cm5zIiwgImlkIl0KICAgIGRhdGFzZXQgPSBkYXRhc2V0LnJlbW92ZV9jb2x1bW5zKAogICAgICAgIFtjIGZvciBjIGluIHJlbW92ZV9jb2xzIGlmIGMgaW4gZGF0YXNldC5jb2x1bW5fbmFtZXNdCiAgICApCiAgICByZXR1cm4gZGF0YXNldAoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIE1ldHJpY3MKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBfc29mdG1heCh4OiBucC5uZGFycmF5LCBheGlzOiBpbnQgPSAtMSkgLT4gbnAubmRhcnJheToKICAgICIiIuaVsOWAvOeos+WumueahCBzb2Z0bWF4IOWunueOsO+8iOmBv+WFjeW8leWFpSBzY2lweSDkvp3otZbvvInjgIIiIiIKICAgIGVfeCA9IG5wLmV4cCh4IC0gbnAubWF4KHgsIGF4aXM9YXhpcywga2VlcGRpbXM9VHJ1ZSkpCiAgICByZXR1cm4gZV94IC8gZV94LnN1bShheGlzPWF4aXMsIGtlZXBkaW1zPVRydWUpCgoKZGVmIGNvbXB1dGVfbWV0cmljcyhldmFsX3ByZWQpIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiIKICAgIFRyYWluZXIg55qEIGNvbXB1dGVfbWV0cmljcyDlm57osIPjgIIKCiAgICDorqHnrpc6CiAgICAgIC0gbG9nX2xvc3M6IOamgueOh+e7j+ijgeWJquWQjuiuoeeul++8jOmBv+WFjSBsb2coMCkg5a+86Ie05p6B56uv5YC8CiAgICAgIC0gYWNjdXJhY3k6IOWIhuexu+WHhuehrueOhwogICAgIiIiCiAgICBsb2dpdHMsIGxhYmVscyA9IGV2YWxfcHJlZAoKICAgICMg5aSE55CG5Y+v6IO955qEIE5hTu+8iENQVSDmqKHlvI/miJbmlbDlgLzkuI3nqLPlrprml7blj6/og73lh7rnjrDvvIkKICAgIG5hbl9tYXNrID0gbnAuaXNuYW4obG9naXRzKS5hbnkoYXhpcz0tMSkKICAgIGlmIG5hbl9tYXNrLmFueSgpOgogICAgICAgIGxvZ2l0cyA9IG5wLmNvcHkobG9naXRzKQogICAgICAgIGxvZ2l0c1tuYW5fbWFza10gPSAwLjAgICMgTmFOIOihjOabv+aNouS4uuWdh+WMgOWIhuW4gwoKICAgICMgbG9naXRzIOKGkiDmpoLnjocKICAgIHByb2JzID0gX3NvZnRtYXgobG9naXRzLCBheGlzPS0xKQoKICAgICMg5qaC546H6KOB5YmqICjlr7kgbG9nX2xvc3Mg6K+E5YiG6Z2e5bi46YeN6KaBKQogICAgZXBzID0gMWUtNwogICAgcHJvYnMgPSBucC5jbGlwKHByb2JzLCBlcHMsIDEuMCAtIGVwcykKICAgIHByb2JzID0gcHJvYnMgLyBwcm9icy5zdW0oYXhpcz0xLCBrZWVwZGltcz1UcnVlKQoKICAgIGxvZ2xvc3MgPSBmbG9hdChsb2dfbG9zcyhsYWJlbHMsIHByb2JzLCBsYWJlbHM9WzAsIDEsIDJdKSkKICAgIHByZWRzID0gbnAuYXJnbWF4KGxvZ2l0cywgYXhpcz0tMSkKICAgIGFjYyA9IGZsb2F0KGFjY3VyYWN5X3Njb3JlKGxhYmVscywgcHJlZHMpKQoKICAgIHJldHVybiB7ImxvZ19sb3NzIjogbG9nbG9zcywgImFjY3VyYWN5IjogYWNjfQo=",
    "hf.py": "IiIiCuaooeWei+WKoOi9veaooeWdlyDigJQg6LSf6LSjIHRva2VuaXplciDlkowgUUxvUkEg5YiG57G75qih5Z6L55qE5Yqg6L2944CCCgrmoLjlv4PnrZbnlaXvvJoKICAxLiBUb2tlbml6ZXI6IOWKoOi9veWQjuajgOafpSBwYWRfdG9rZW7vvIzoi6XnvLrlpLHliJnorr7kuLogZW9zX3Rva2VuCiAgMi4g5qih5Z6LOiDpgJrov4cgQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAzLiBMb1JBOiDkvb/nlKggUEVGVCDnmoQgTG9yYUNvbmZpZyDms6jlhaUgYWRhcHRlcu+8jOWQjOaXtuS/neWtmOWIhuexu+WktCAoc2NvcmUpCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCmZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgVGFza1R5cGUsIGdldF9wZWZ0X21vZGVsLCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoCiAgICBBdXRvQ29uZmlnLAogICAgQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbiwKICAgIEF1dG9Ub2tlbml6ZXIsCiAgICBCaXRzQW5kQnl0ZXNDb25maWcsCikKCmZyb20gYXJlbmFfcmFua2VyLmNvbmZpZyBpbXBvcnQgTW9kZWxDb25maWcsIE5VTV9MQUJFTFMKCkxPR0dFUiA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJhcmVuYV9yYW5rZXIuaGYiKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOi+heWKqeWHveaVsAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIF9kZXNjcmliZV9tb2RlbF9zb3VyY2UobW9kZWxfbmFtZTogc3RyKSAtPiBzdHI6CiAgICAiIiLmj4/ov7DmqKHlnovmnaXmupDvvIzmlrnkvr/mjpLmn6XliqDovb3pl67popjjgIIiIiIKICAgIHNvdXJjZV9wYXRoID0gUGF0aChtb2RlbF9uYW1lKS5leHBhbmR1c2VyKCkKICAgIGlmIHNvdXJjZV9wYXRoLmV4aXN0cygpOgogICAgICAgIGNvbmZpZ19wYXRoID0gc291cmNlX3BhdGggLyAiY29uZmlnLmpzb24iCiAgICAgICAgaWYgY29uZmlnX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBmIuajgOa1i+WIsOacrOWcsOaooeWei+ebruW9le+8mntzb3VyY2VfcGF0aH0iCiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZiLmo4DmtYvliLDlkIzlkI3mnKzlnLDnm67lvZUge3NvdXJjZV9wYXRofe+8jOS9hue8uuWwkSBjb25maWcuanNvbuOAgiIKICAgICAgICAgICAgInRyYW5zZm9ybWVycyDkvJrmiorlroPlvZPmiJDmnKzlnLDmqKHlnovot6/lvoTlubbnm7TmjqXliqDovb3lpLHotKXjgIIiCiAgICAgICAgKQogICAgaWYgIi8iIGluIG1vZGVsX25hbWU6CiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZiJge21vZGVsX25hbWV9YCDlsIbooqvlvZPkvZwgSHVnZ2luZ0ZhY2Ug5LuT5bqTIElE44CCIgogICAgICAgICAgICAi6aaW5qyh6L+Q6KGM6ZyA6IGU572R5LiL6L2977yb56a757q/546v5aKD6K+36aKE5LiL6L295Yiw5pys5Zyw5YaN5L+u5pS5IG1vZGVsX25hbWXjgIIiCiAgICAgICAgKQogICAgcmV0dXJuIGYiYHttb2RlbF9uYW1lfWAg5pei5LiN5piv5pys5Zyw55uu5b2V77yM5Lmf5LiN5piv5qCH5YeGIEh1Z2dpbmdGYWNlIOS7k+W6kyBJROOAgiIKCgpkZWYgX2dldF9sb2NhbF9yYW5rKCkgLT4gaW50IHwgTm9uZToKICAgICIiIuivu+WPliB0b3JjaC5kaXN0cmlidXRlZCDms6jlhaXnmoQgTE9DQUxfUkFOS+OAgiIiIgogICAgcmF3ID0gb3MuZW52aXJvbi5nZXQoIkxPQ0FMX1JBTksiKQogICAgaWYgcmF3IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHRyeToKICAgICAgICByZXR1cm4gaW50KHJhdykKICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgIExPR0dFUi53YXJuaW5nKCLlv73nlaXpnZ7ms5UgTE9DQUxfUkFOSz0lciIsIHJhdykKICAgICAgICByZXR1cm4gTm9uZQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIFRva2VuaXplcgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGxvYWRfdG9rZW5pemVyKGNvbmZpZzogTW9kZWxDb25maWcpOgogICAgIiIiCiAgICDliqDovb0gdG9rZW5pemVyIOW5tumFjee9riBwYWRfdG9rZW7jgIIKCiAgICBRd2VuIOezu+WIl+aooeWei+mAmuW4uOayoeaciem7mOiupCBwYWRfdG9rZW7vvIzov5nph4zlsIblhbborr7kuLogZW9zX3Rva2Vu44CCCiAgICDlkIzml7borr7nva4gcGFkZGluZ19zaWRlPSJsZWZ0Iu+8iGRlY29kZXItb25seSDmqKHlnovlgZrliIbnsbvml7bnmoTmjqjojZDlgZrms5XvvInjgIIKICAgICIiIgogICAgdHJ5OgogICAgICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBjb25maWcubW9kZWxfbmFtZSwKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9Y29uZmlnLmxvY2FsX2ZpbGVzX29ubHksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb0gdG9rZW5pemVyIOWksei0pToge2V4Y31cbiIKICAgICAgICAgICAgZiJ7X2Rlc2NyaWJlX21vZGVsX3NvdXJjZShjb25maWcubW9kZWxfbmFtZSl9IgogICAgICAgICkgZnJvbSBleGMKCiAgICAjIOiuvue9riBwYWRfdG9rZW7vvIhRd2VuIOmAmuW4uOayoeaciem7mOiupOWAvO+8iQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBMT0dHRVIuaW5mbygKICAgICAgICAgICAgIlRva2VuaXplciDnvLrlsJEgcGFkX3Rva2Vu77yM5bey6K6+5Li6IGVvc190b2tlbjogJyVzJyAoaWQ9JXMpIiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCwKICAgICAgICApCgogICAgIyBkZWNvZGVyLW9ubHkg5qih5Z6L5YGa5YiG57G75pe277yM5bem5L6n5aGr5YWF5Y+v6YG/5YWN5pyA5ZCO5LiA5LiqIHRva2VuIOS4uiBwYWQKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCiAgICByZXR1cm4gdG9rZW5pemVyCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg6YeP5YyW6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2JuYl9jb25maWcoY29uZmlnOiBNb2RlbENvbmZpZykgLT4gQml0c0FuZEJ5dGVzQ29uZmlnIHwgTm9uZToKICAgICIiIuaehOW7uiBCaXRzQW5kQnl0ZXNDb25maWcg55So5LqOIDQtYml0IFFMb1JBIOmHj+WMluOAgiIiIgogICAgaWYgbm90IGNvbmZpZy5sb2FkX2luXzRiaXQ6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgICMgNC1iaXQg6YeP5YyW6ZyA6KaBIENVREHvvJtDUFUg5qih5byP5LiL6Lez6L+HCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBMT0dHRVIud2FybmluZygi5pyq5qOA5rWL5YiwIENVREHvvIzot7Pov4cgNC1iaXQg6YeP5YyW77yI5bCG5LulIGZsb2F0MzIg5Yqg6L295qih5Z6L77yJIikKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPWNvbmZpZy5ibmJfNGJpdF9xdWFudF90eXBlLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50PWNvbmZpZy5ibmJfNGJpdF91c2VfZG91YmxlX3F1YW50LAogICAgKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIExvUkEg6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2xvcmFfY29uZmlnKGNvbmZpZzogTW9kZWxDb25maWcpIC0+IExvcmFDb25maWc6CiAgICAiIiIKICAgIOaehOW7uiBMb1JBIOmAgumFjeWZqOmFjee9ruOAggoKICAgIOWFs+mUruWPguaVsOivtOaYjjoKICAgICAgLSB0YXJnZXRfbW9kdWxlczog5a+55omA5pyJIGF0dGVudGlvbiArIEZGTiDmipXlvbHlsYLms6jlhaUgTG9SQQogICAgICAtIG1vZHVsZXNfdG9fc2F2ZT1bInNjb3JlIl06IOWIhuexu+WktCAoc2NvcmUpIOW/hemhu+WFqOmHj+iuree7g+W5tuS/neWtmO+8jAogICAgICAgIOWQpuWImemaj+acuuWIneWni+WMlueahOWIhuexu+WktOS4jeS8muiiq+aMgeS5heWMlgogICAgICAtIHRhc2tfdHlwZT1TRVFfQ0xTOiDlkYror4kgUEVGVCDov5nmmK/kuIDkuKrluo/liJfliIbnsbvku7vliqEKICAgICIiIgogICAgcmV0dXJuIExvcmFDb25maWcoCiAgICAgICAgcj1jb25maWcubG9yYV9yLAogICAgICAgIGxvcmFfYWxwaGE9Y29uZmlnLmxvcmFfYWxwaGEsCiAgICAgICAgdGFyZ2V0X21vZHVsZXM9Y29uZmlnLmxvcmFfdGFyZ2V0X21vZHVsZXMsCiAgICAgICAgbG9yYV9kcm9wb3V0PWNvbmZpZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgYmlhcz1jb25maWcubG9yYV9iaWFzLAogICAgICAgIHRhc2tfdHlwZT1UYXNrVHlwZS5TRVFfQ0xTLAogICAgICAgIG1vZHVsZXNfdG9fc2F2ZT1jb25maWcubG9yYV9tb2R1bGVzX3RvX3NhdmUsCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Yqg6L295YiG57G75qih5Z6LIChRTG9SQSkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX21vZGVsKGNvbmZpZzogTW9kZWxDb25maWcsIHRva2VuaXplcj1Ob25lKToKICAgICIiIgogICAg5Yqg6L29IFFMb1JBIOWIhuexu+aooeWei++8jOWujOaVtOa1geeoi++8mgogICAgICAxLiDkvb/nlKggQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAgICAgMi4g6YCa6L+HIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24g5pu/5o2i6K+t6KiA5bu65qih5aS05Li65YiG57G75aS0IChudW1fbGFiZWxzPTMpCiAgICAgIDMuIOS9v+eUqCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nIOWHhuWkh+mHj+WMluaooeWeiwogICAgICA0LiDms6jlhaUgTG9SQSBhZGFwdGVyCgogICAgUmV0dXJuczoKICAgICAgICBQRUZUIOWMheijheWQjueahOaooeWei++8iOiLpSB1c2VfbG9yYT1UcnVl77yJ77yM5ZCm5YiZ5Y6f5aeL5qih5Z6L44CCCiAgICAiIiIKICAgIGJuYl9jb25maWcgPSBfYnVpbGRfYm5iX2NvbmZpZyhjb25maWcpCiAgICBsb2NhbF9yYW5rID0gX2dldF9sb2NhbF9yYW5rKCkKCiAgICAjIFF3ZW4g57O75YiX6YOo5YiG6YWN572u5Lya5bim5bWM5aWXIHRleHRfY29uZmln77yM6ZyA6KaB56Gu5L+dIG51bV9sYWJlbHMg5q2j56Gu5Lyg5pKt44CCCiAgICB0cnk6CiAgICAgICAgbW9kZWxfY29uZmlnID0gQXV0b0NvbmZpZy5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIGNvbmZpZy5tb2RlbF9uYW1lLAogICAgICAgICAgICBudW1fbGFiZWxzPU5VTV9MQUJFTFMsCiAgICAgICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgICAgIGNhY2hlX2Rpcj1jb25maWcuY2FjaGVfZGlyLAogICAgICAgICAgICBsb2NhbF9maWxlc19vbmx5PWNvbmZpZy5sb2NhbF9maWxlc19vbmx5LAogICAgICAgICkKICAgIGV4Y2VwdCBPU0Vycm9yIGFzIGV4YzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYi5Yqg6L295qih5Z6L6YWN572u5aSx6LSlOiB7ZXhjfVxuIgogICAgICAgICAgICBmIntfZGVzY3JpYmVfbW9kZWxfc291cmNlKGNvbmZpZy5tb2RlbF9uYW1lKX0iCiAgICAgICAgKSBmcm9tIGV4YwoKICAgICMg5p+Q5LqbIFF3ZW4g6YWN572u5LiN5Lya6Ieq5Yqo5oqKIG51bV9sYWJlbHMg5Lyg5pKt5YiwIHRleHRfY29uZmlnCiAgICBpZiBoYXNhdHRyKG1vZGVsX2NvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgbW9kZWxfY29uZmlnLnRleHRfY29uZmlnLm51bV9sYWJlbHMgPSBOVU1fTEFCRUxTCiAgICAgICAgTE9HR0VSLmluZm8oIuW3suaJi+WKqOWwhiBudW1fbGFiZWxzPSVzIOS8oOaSreWIsCB0ZXh0X2NvbmZpZyIsIE5VTV9MQUJFTFMpCgogICAgaWYgbG9jYWxfcmFuayBpcyBub3QgTm9uZSBhbmQgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICB0b3JjaC5jdWRhLnNldF9kZXZpY2UobG9jYWxfcmFuaykKICAgICAgICBMT0dHRVIuaW5mbygi5YiG5biD5byP6K6t57uDOiDlvZPliY3ov5vnqIvnu5HlrprliLAgY3VkYTolcyIsIGxvY2FsX3JhbmspCgogICAgZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSB0b3JjaC5mbG9hdDMyCiAgICBtb2RlbF9rd2FyZ3MgPSBkaWN0KAogICAgICAgIHByZXRyYWluZWRfbW9kZWxfbmFtZV9vcl9wYXRoPWNvbmZpZy5tb2RlbF9uYW1lLAogICAgICAgIGNvbmZpZz1tb2RlbF9jb25maWcsCiAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1ibmJfY29uZmlnLAogICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgbG9jYWxfZmlsZXNfb25seT1jb25maWcubG9jYWxfZmlsZXNfb25seSwKICAgICAgICB0b3JjaF9kdHlwZT1kdHlwZSwKICAgICkKICAgICMgUUxvUkEgKyBERFAg6ZyA6KaB56Gu5L+d5q+P5LiqIHJhbmsg5Y+q5oqK6YeP5YyW5qih5Z6L5Yqg6L295Yiw5pys5ZywIEdQVeOAggogICAgaWYgYm5iX2NvbmZpZyBpcyBub3QgTm9uZSBhbmQgbG9jYWxfcmFuayBpcyBub3QgTm9uZToKICAgICAgICBtb2RlbF9rd2FyZ3NbImRldmljZV9tYXAiXSA9IHsiIjogbG9jYWxfcmFua30KICAgIHRyeToKICAgICAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKCoqbW9kZWxfa3dhcmdzKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb3mqKHlnovlpLHotKU6IHtleGN9XG4iCiAgICAgICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UoY29uZmlnLm1vZGVsX25hbWUpfSIKICAgICAgICApIGZyb20gZXhjCgogICAgIyDlr7npvZAgcGFkX3Rva2VuX2lkCiAgICBpZiB0b2tlbml6ZXIgaXMgbm90IE5vbmUgYW5kIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwuY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsLmNvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgICAgIG1vZGVsLmNvbmZpZy50ZXh0X2NvbmZpZy5wYWRfdG9rZW5faWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCgogICAgaWYgbm90IGNvbmZpZy51c2VfbG9yYToKICAgICAgICByZXR1cm4gbW9kZWwKCiAgICAjIFFMb1JBIOeJueacieatpemqpDog5YeG5aSH6YeP5YyW5qih5Z6L5Lul6YCC6YWN6K6t57uD77yI5LuF5Zyo5a6e6ZmF6YeP5YyW5pe277yJCiAgICBpZiBjb25maWcubG9hZF9pbl80Yml0IGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIG1vZGVsID0gcHJlcGFyZV9tb2RlbF9mb3Jfa2JpdF90cmFpbmluZygKICAgICAgICAgICAgbW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUKICAgICAgICApCgogICAgbG9yYV9jb25maWcgPSBfYnVpbGRfbG9yYV9jb25maWcoY29uZmlnKQogICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgbG9yYV9jb25maWcpCgogICAgIyDmiZPljbDlj6/orq3nu4Plj4LmlbDnu5/orqEKICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKCiAgICByZXR1cm4gbW9kZWwK",
    "modeling.py": "IiIiCuaooeWei+WumuS5ieaooeWdl+OAggoK5b2T5YmN54mI5pys5L2/55SoIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24gKyBRTG9SQe+8jArmqKHlnovnmoTliqDovb3lkozphY3nva7lt7Lnp7voh7MgaGYucHnjgILmraTmqKHlnZfkv53nlZnkuLrljaDkvY3vvIzkvpvmnKrmnaXmianlsZXkvb/nlKjjgIIKIiIiCg==",
    "dummy_data.py": "IiIiCuWBh+aVsOaNrueUn+aIkOaooeWdlyDigJQg55Sf5oiQIGR1bW15IHRyYWluLmNzdiDlkowgdGVzdC5jc3Yg55So5LqO5pys5Zyw5rWL6K+V44CCCgrkvb/nlKjmlrnms5XvvJoKICB1diBydW4gYXJlbmEtZHVtbXktZGF0YSAgICAgICAgICAgICAgICAgICAgIyDpu5jorqQgMTAwIOadoeiuree7gyArIDIwIOadoea1i+ivlQogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tbi10cmFpbiA1MCAtLW4tdGVzdCAxMAogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tb3V0cHV0LWRpciAuL2RhdGEKCuS5n+WPr+S7peWcqCBQeXRob24g5Lit55u05o6l6LCD55So77yaCiAgZnJvbSBhcmVuYV9yYW5rZXIuZHVtbXlfZGF0YSBpbXBvcnQgZ2VuZXJhdGVfZHVtbXlfZGF0YQogIGdlbmVyYXRlX2R1bW15X2RhdGEobl90cmFpbj0xMDAsIG5fdGVzdD0yMCkKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHJhbmRvbQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgojIOWBh+aPkOmXruWIl+ihqApfUFJPTVBUUyA9IFsKICAgICJXaGF0IGlzIG1hY2hpbmUgbGVhcm5pbmc/IEV4cGxhaW4gaXQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV3JpdGUgYSBQeXRob24gZnVuY3Rpb24gdGhhdCBzb3J0cyBhIGxpc3QgdXNpbmcgbWVyZ2Ugc29ydC4iLAogICAgIkV4cGxhaW4gdGhlIHRoZW9yeSBvZiByZWxhdGl2aXR5IHRvIGEgMTAteWVhci1vbGQuIiwKICAgICJIb3cgZG9lcyBwaG90b3N5bnRoZXNpcyB3b3JrPyBQbGVhc2UgYmUgZGV0YWlsZWQuIiwKICAgICJXaGF0IGFyZSB0aGUgbWFpbiBkaWZmZXJlbmNlcyBiZXR3ZWVuIFRDUCBhbmQgVURQPyIsCiAgICAiU3VtbWFyaXplIHRoZSBwbG90IG9mIFJvbWVvIGFuZCBKdWxpZXQuIiwKICAgICJXcml0ZSBhIHBvZW0gYWJvdXQgdGhlIG9jZWFuIGF0IHN1bnNldC4iLAogICAgIkV4cGxhaW4gcXVhbnR1bSBlbnRhbmdsZW1lbnQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBGcmFuY2UgYW5kIHdoYXQgaXMgaXQga25vd24gZm9yPyIsCiAgICAiSG93IGRvIG5ldXJhbCBuZXR3b3JrcyBsZWFybj8gRXhwbGFpbiBiYWNrcHJvcGFnYXRpb24uIiwKICAgICJDb21wYXJlIFB5dGhvbiBhbmQgUnVzdCBmb3Igc3lzdGVtcyBwcm9ncmFtbWluZy4iLAogICAgIldoYXQgYXJlIHRoZSBiZW5lZml0cyBvZiBtZWRpdGF0aW9uPyIsCiAgICAiRGVzY3JpYmUgaG93IGEgQ1BVIGV4ZWN1dGVzIGluc3RydWN0aW9ucy4iLAogICAgIldyaXRlIGEgU1FMIHF1ZXJ5IHRvIGZpbmQgZHVwbGljYXRlIGVtYWlscyBpbiBhIHRhYmxlLiIsCiAgICAiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIHN1cGVydmlzZWQgYW5kIHVuc3VwZXJ2aXNlZCBsZWFybmluZy4iLApdCgpfUkVTUE9OU0VfVEVNUExBVEVTX0EgPSBbCiAgICAiVGhhdCdzIGEgZ3JlYXQgcXVlc3Rpb24hIExldCBtZSBleHBsYWluLiB7dG9waWN9IGlzIGZ1bmRhbWVudGFsbHkgYWJvdXQgIgogICAgInVuZGVyc3RhbmRpbmcgcGF0dGVybnMgaW4gZGF0YS4gVGhlIGtleSBpbnNpZ2h0IGlzIHRoYXQgd2UgY2FuIHVzZSBtYXRoZW1hdGljYWwgIgogICAgIm1vZGVscyB0byBhcHByb3hpbWF0ZSBjb21wbGV4IHJlbGF0aW9uc2hpcHMuIEhlcmUncyBhIG1vcmUgZGV0YWlsZWQgYnJlYWtkb3duOiAiCiAgICAiRmlyc3QsIHdlIG5lZWQgdG8gY29sbGVjdCByZWxldmFudCBkYXRhLiBUaGVuLCB3ZSBwcmVwcm9jZXNzIGl0IHRvIHJlbW92ZSBub2lzZS4gIgogICAgIkZpbmFsbHksIHdlIHRyYWluIG91ciBtb2RlbCBhbmQgZXZhbHVhdGUgaXRzIHBlcmZvcm1hbmNlLiIsCiAgICAiU3VyZSEgSGVyZSdzIG15IHRha2Ugb24gdGhpcy4ge3RvcGljfSBpbnZvbHZlcyBzZXZlcmFsIGltcG9ydGFudCBjb25jZXB0cy4gIgogICAgIlRoZSBtb3N0IGZ1bmRhbWVudGFsIG9uZSBpcyB0aGF0IGxlYXJuaW5nIGhhcHBlbnMgdGhyb3VnaCBpdGVyYXRpdmUgb3B0aW1pemF0aW9uLiAiCiAgICAiV2Ugc3RhcnQgd2l0aCBhIHJhbmRvbSBndWVzcyBhbmQgZ3JhZHVhbGx5IGltcHJvdmUgaXQgYmFzZWQgb24gZmVlZGJhY2sgZnJvbSB0aGUgZGF0YS4iLAogICAgIkdyZWF0IHF1ZXN0aW9uLiBJbiBzaW1wbGUgdGVybXMsIHt0b3BpY30gaXMgbGlrZSB0ZWFjaGluZyBhIGNvbXB1dGVyIHRvIHJlY29nbml6ZSAiCiAgICAicGF0dGVybnMuIEltYWdpbmUgc2hvd2luZyBhIGNoaWxkIHRob3VzYW5kcyBvZiBwaWN0dXJlcyBvZiBjYXRzIGFuZCBkb2dzIC0gZXZlbnR1YWxseSAiCiAgICAidGhleSBsZWFybiB0byB0ZWxsIHRoZW0gYXBhcnQuIFRoYXQncyBlc3NlbnRpYWxseSB3aGF0IGhhcHBlbnMgaW4gdGhpcyBwcm9jZXNzLiIsCl0KCl9SRVNQT05TRV9URU1QTEFURVNfQiA9IFsKICAgICJUaGFua3MgZm9yIGFza2luZyEge3RvcGljfSBpcyBhY3R1YWxseSBzaW1wbGVyIHRoYW4gbW9zdCBwZW9wbGUgdGhpbmsuICIKICAgICJBdCBpdHMgY29yZSwgaXQncyBhYm91dCBmaW5kaW5nIHRoZSBiZXN0IGZ1bmN0aW9uIHRoYXQgbWFwcyBpbnB1dHMgdG8gb3V0cHV0cy4gIgogICAgIldlIGRvIHRoaXMgYnkgbWluaW1pemluZyBhIGxvc3MgZnVuY3Rpb24gdXNpbmcgZ3JhZGllbnQtYmFzZWQgb3B0aW1pemF0aW9uLiAiCiAgICAiVGhlIGJlYXV0eSBvZiB0aGlzIGFwcHJvYWNoIGlzIGl0cyBnZW5lcmFsaXR5IC0gaXQgd29ya3MgYWNyb3NzIG1hbnkgZG9tYWlucy4iLAogICAgIkxldCBtZSBicmVhayB0aGlzIGRvd24uIHt0b3BpY30gY2FuIGJlIHVuZGVyc3Rvb2QgdGhyb3VnaCBhIHNpbXBsZSBhbmFsb2d5LiAiCiAgICAiVGhpbmsgb2YgaXQgYXMgYSByZWNpcGUgLSB5b3UgaGF2ZSBpbmdyZWRpZW50cyAoZGF0YSksIGluc3RydWN0aW9ucyAoYWxnb3JpdGhtKSwgIgogICAgImFuZCBhIGZpbmFsIGRpc2ggKHByZWRpY3Rpb25zKS4gVGhlIHF1YWxpdHkgb2YgZWFjaCBpbmdyZWRpZW50IG1hdHRlcnMsIGJ1dCBzbyBkb2VzICIKICAgICJob3cgeW91IGNvbWJpbmUgdGhlbS4iLAogICAgIkknZCBiZSBoYXBweSB0byBleHBsYWluLiB7dG9waWN9IGlzIGEgZmFzY2luYXRpbmcgYXJlYS4gVGhlIGJhc2ljIGlkZWEgaXMgIgogICAgInRoYXQgd2UgY2FuIHVzZSBzdGF0aXN0aWNzIGFuZCBjb21wdXRhdGlvbiB0byBtYWtlIHByZWRpY3Rpb25zIGFib3V0IHRoZSB3b3JsZC4gIgogICAgIlRoZSBrZXkgY2hhbGxlbmdlIGlzIGdlbmVyYWxpemF0aW9uIC0gbWFraW5nIHN1cmUgb3VyIG1vZGVsIHdvcmtzIG9uIG5ldywgdW5zZWVuIGRhdGEuIiwKXQoKCmRlZiBnZW5lcmF0ZV9kdW1teV9kYXRhKAogICAgb3V0cHV0X2Rpcjogc3RyID0gIi4iLAogICAgbl90cmFpbjogaW50ID0gMTAwLAogICAgbl90ZXN0OiBpbnQgPSAyMCwKICAgIHNlZWQ6IGludCA9IDQyLAopIC0+IHR1cGxlW1BhdGgsIFBhdGhdOgogICAgIiIiCiAgICDnlJ/miJDnlKjkuo7mnKzlnLDmtYvor5XnmoTlgYfmlbDmja7jgIIKCiAgICDmoIfnrb7liIbluIPlpKfoh7TkuLrvvJpB6IOcIDQwJSwgQuiDnCA0MCUsIOW5s+WxgCAyMCXvvIjmqKHmi5/nnJ/lrp7mlbDmja7liIbluIPvvInjgIIKCiAgICBSZXR1cm5zOgogICAgICAgICh0cmFpbl9wYXRoLCB0ZXN0X3BhdGgpCiAgICAiIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShzZWVkKQogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG91dCA9IFBhdGgob3V0cHV0X2RpcikKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyAtLS0tIOiuree7g+mbhiAtLS0tCiAgICB0cmFpbl9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdHJhaW4pOgogICAgICAgIHByb21wdCA9IF9QUk9NUFRTW2kgJSBsZW4oX1BST01QVFMpXQogICAgICAgIHRvcGljID0gcHJvbXB0LnNwbGl0KCI/IilbMF0uc3BsaXQoIi4iKVswXS5zdHJpcCgpCgogICAgICAgIHJlc3BfYSA9IHJhbmRvbS5jaG9pY2UoX1JFU1BPTlNFX1RFTVBMQVRFU19BKS5mb3JtYXQodG9waWM9dG9waWMpCiAgICAgICAgcmVzcF9iID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0IpLmZvcm1hdCh0b3BpYz10b3BpYykKCiAgICAgICAgbGFiZWwgPSBybmcuY2hvaWNlKFswLCAxLCAyXSwgcD1bMC40LCAwLjQsIDAuMl0pCiAgICAgICAgdHJhaW5fcm93cy5hcHBlbmQoewogICAgICAgICAgICAiaWQiOiAxMDAwMDAgKyBpLAogICAgICAgICAgICAibW9kZWxfYSI6IGYibW9kZWxfeF97cm5nLnJhbmRpbnQoMSwgNSl9IiwKICAgICAgICAgICAgIm1vZGVsX2IiOiBmIm1vZGVsX3lfe3JuZy5yYW5kaW50KDEsIDUpfSIsCiAgICAgICAgICAgICJwcm9tcHQiOiBwcm9tcHQsCiAgICAgICAgICAgICJyZXNwb25zZV9hIjogcmVzcF9hLAogICAgICAgICAgICAicmVzcG9uc2VfYiI6IHJlc3BfYiwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9hIjogMSBpZiBsYWJlbCA9PSAwIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9iIjogMSBpZiBsYWJlbCA9PSAxIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl90aWUiOiAxIGlmIGxhYmVsID09IDIgZWxzZSAwLAogICAgICAgIH0pCgogICAgdHJhaW5fcGF0aCA9IG91dCAvICJ0cmFpbi5jc3YiCiAgICBwZC5EYXRhRnJhbWUodHJhaW5fcm93cykudG9fY3N2KHRyYWluX3BhdGgsIGluZGV4PUZhbHNlKQoKICAgICMgLS0tLSDmtYvor5Xpm4YgLS0tLQogICAgdGVzdF9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdGVzdCk6CiAgICAgICAgcHJvbXB0ID0gX1BST01QVFNbaSAlIGxlbihfUFJPTVBUUyldCiAgICAgICAgdG9waWMgPSBwcm9tcHQuc3BsaXQoIj8iKVswXS5zcGxpdCgiLiIpWzBdLnN0cmlwKCkKCiAgICAgICAgcmVzcF9hID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0EpLmZvcm1hdCh0b3BpYz10b3BpYykKICAgICAgICByZXNwX2IgPSByYW5kb20uY2hvaWNlKF9SRVNQT05TRV9URU1QTEFURVNfQikuZm9ybWF0KHRvcGljPXRvcGljKQoKICAgICAgICB0ZXN0X3Jvd3MuYXBwZW5kKHsKICAgICAgICAgICAgImlkIjogMjAwMDAwICsgaSwKICAgICAgICAgICAgInByb21wdCI6IHByb21wdCwKICAgICAgICAgICAgInJlc3BvbnNlX2EiOiByZXNwX2EsCiAgICAgICAgICAgICJyZXNwb25zZV9iIjogcmVzcF9iLAogICAgICAgIH0pCgogICAgdGVzdF9wYXRoID0gb3V0IC8gInRlc3QuY3N2IgogICAgcGQuRGF0YUZyYW1lKHRlc3Rfcm93cykudG9fY3N2KHRlc3RfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgcHJpbnQoZiLlt7LnlJ/miJDorq3nu4Ppm4Y6IHt0cmFpbl9wYXRofSAoe25fdHJhaW59IOadoSkiKQogICAgcHJpbnQoZiLlt7LnlJ/miJDmtYvor5Xpm4Y6IHt0ZXN0X3BhdGh9ICh7bl90ZXN0fSDmnaEpIikKICAgIHJldHVybiB0cmFpbl9wYXRoLCB0ZXN0X3BhdGgKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0i55Sf5oiQ5YGH5pWw5o2u55So5LqO5pys5Zyw5rWL6K+V44CCIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSIuIiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye655uu5b2VIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10cmFpbiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMCwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6K6t57uD6ZuG6KGM5pWwIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10ZXN0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iua1i+ivlembhuihjOaVsCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZ2VuZXJhdGVfZHVtbXlfZGF0YSgKICAgICAgICBvdXRwdXRfZGlyPWFyZ3Mub3V0cHV0X2RpciwKICAgICAgICBuX3RyYWluPWFyZ3Mubl90cmFpbiwKICAgICAgICBuX3Rlc3Q9YXJncy5uX3Rlc3QsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "train.py": "...",
    "predict.py": "IiIiCuaOqOeQhuiEmuacrCDigJQg5Yqg6L296K6t57uD5aW955qEIFFMb1JBIGFkYXB0ZXLvvIzlr7nmtYvor5Xpm4bnlJ/miJAgc3VibWlzc2lvbi5jc3bjgIIKCua1geeoi++8mgogIDEuIOmHjeaWsOWKoOi9veWfuuW6p+aooeWeiyAoNC1iaXQg6YeP5YyWKQogIDIuIOWKoOi9veS/neWtmOeahCBMb1JBIGFkYXB0ZXIgKyDliIbnsbvlpLQKICAzLiDpgY3ljobmtYvor5Xpm4bvvIznlJ/miJDkuInliIbnsbvmpoLnjocKICA0LiDmpoLnjofoo4HliaogKyDlvZLkuIDljJblkI7lhpnlhaUgc3VibWlzc2lvbi5jc3YKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoCiAgICBBdXRvQ29uZmlnLAogICAgQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbiwKICAgIEF1dG9Ub2tlbml6ZXIsCiAgICBCaXRzQW5kQnl0ZXNDb25maWcsCiAgICBEYXRhQ29sbGF0b3JXaXRoUGFkZGluZywKKQoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBJRF9UT19MQUJFTCwgTlVNX0xBQkVMUywgbG9hZF9jb25maWcKZnJvbSBhcmVuYV9yYW5rZXIuZGF0YSBpbXBvcnQgYnVpbGRfZGF0YXNldCwgbG9hZF9hbmRfcHJlcHJvY2VzcwoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyZW5hX3Jhbmtlci5wcmVkaWN0IikKUFJPQkFCSUxJVFlfRVBTSUxPTiA9IDFlLTYKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDTEkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IkdlbmVyYXRlIHN1Ym1pc3Npb24gd2l0aCB0cmFpbmVkIFFMb1JBIEFyZW5hIHJhbmtlci4iCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQtZGlyIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iuiuree7g+S6p+eJqeebruW9lSAo5YyF5ZCrIG1vZGVsLywgdG9rZW5pemVyLywgY29uZmlnLnlhbWwpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD0iLiIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9InRlc3QuY3N2IOaJgOWcqOebruW9lSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1wYXRoIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye6IHN1Ym1pc3Npb24uY3N2IOi3r+W+hCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD00LAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSLmjqjnkIYgYmF0Y2ggc2l6ZSIpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOW3peWFt+WHveaVsAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIHNldHVwX2xvZ2dpbmcoKSAtPiBOb25lOgogICAgbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgICAgICBsZXZlbD1sb2dnaW5nLklORk8sCiAgICAgICAgZm9ybWF0PSIlKGFzY3RpbWUpcyB8ICUobGV2ZWxuYW1lKXMgfCAlKG1lc3NhZ2UpcyIsCiAgICAgICAgZGF0ZWZtdD0iJUg6JU06JVMiLAogICAgKQoKCmRlZiBmb3JtYXRfc2Vjb25kcyhzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgdG90YWwgPSBtYXgoaW50KHNlY29uZHMpLCAwKQogICAgbSwgcyA9IGRpdm1vZCh0b3RhbCwgNjApCiAgICBoLCBtID0gZGl2bW9kKG0sIDYwKQogICAgaWYgaCA+IDA6CiAgICAgICAgcmV0dXJuIGYie2h9aCB7bX1tIHtzfXMiCiAgICBpZiBtID4gMDoKICAgICAgICByZXR1cm4gZiJ7bX1tIHtzfXMiCiAgICByZXR1cm4gZiJ7c31zIgoKCmRlZiBkZXNjcmliZV9kZXZpY2UoZGV2aWNlOiB0b3JjaC5kZXZpY2UpIC0+IHN0cjoKICAgIGlmIGRldmljZS50eXBlICE9ICJjdWRhIjoKICAgICAgICByZXR1cm4gIkNQVSIKICAgIG5hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShkZXZpY2UpCiAgICBtZW0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhkZXZpY2UpLnRvdGFsX21lbW9yeSAvIDEwMjQqKjMKICAgIHJldHVybiBmIntuYW1lfSAoe21lbTouMWZ9IEdCKSIKCgpkZWYgbm9ybWFsaXplX3Byb2JhYmlsaXRpZXMoCiAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciwKICAgIGVwc2lsb246IGZsb2F0ID0gUFJPQkFCSUxJVFlfRVBTSUxPTiwKKSAtPiBucC5uZGFycmF5OgogICAgIiIibG9naXRzIOKGkiDoo4HliarlkI7nmoTmpoLnjofvvIjnoa7kv50gbG9nX2xvc3Mg5LiN5Lya54iG54K477yJ44CCIiIiCiAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobG9naXRzLCBkaW09LTEpLmNwdSgpLm51bXB5KCkKICAgIHByb2JzID0gbnAuY2xpcChwcm9icywgZXBzaWxvbiwgMS4wIC0gZXBzaWxvbikKICAgIHByb2JzID0gcHJvYnMgLyBwcm9icy5zdW0oYXhpcz0tMSwga2VlcGRpbXM9VHJ1ZSkKICAgIHJldHVybiBwcm9icwoKCmRlZiBydW5faW5mZXJlbmNlKGxvYWRlcjogRGF0YUxvYWRlciwgbW9kZWwsIGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBucC5uZGFycmF5OgogICAgIiIi5a+55LiA5LiqIERhdGFMb2FkZXIg5omn6KGM5a6M5pW05o6o55CG5bm26L+U5Zue5qaC546H55+p6Zi144CCIiIiCiAgICBhbGxfcHJvYnMgPSBbXQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGJhdGNoIGluIHRxZG0obG9hZGVyLCBkZXNjPSJwcmVkaWN0Iik6CiAgICAgICAgICAgIGJhdGNoID0ge2s6IHYudG8oZGV2aWNlKSBmb3IgaywgdiBpbiBiYXRjaC5pdGVtcygpfQogICAgICAgICAgICBvdXRwdXRzID0gbW9kZWwoKipiYXRjaCkKICAgICAgICAgICAgcHJvYnMgPSBub3JtYWxpemVfcHJvYmFiaWxpdGllcyhvdXRwdXRzLmxvZ2l0cykKICAgICAgICAgICAgYWxsX3Byb2JzLmFwcGVuZChwcm9icykKICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShhbGxfcHJvYnMsIGF4aXM9MCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDliqDovb3mjqjnkIbmqKHlnosKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX2luZmVyZW5jZV9tb2RlbChjaGVja3BvaW50X2RpcjogUGF0aCwgY29uZmlnLCB0b2tlbml6ZXIpOgogICAgIiIiCiAgICDliqDovb3orq3nu4Plpb3nmoQgUUxvUkEg5qih5Z6L55So5LqO5o6o55CG44CCCgogICAg5q2l6aqk77yaCiAgICAgIDEuIOmHjeaWsOWKoOi9veWfuuW6p+aooeWeiyAoNC1iaXQg6YeP5YyWKQogICAgICAyLiDku44gYWRhcHRlciDnm67lvZXliqDovb0gTG9SQSDmnYPph40gKyBzY29yZSDliIbnsbvlpLQKICAgICAgMy4g5a+56b2QIHBhZF90b2tlbl9pZAogICAgIiIiCiAgICBtYyA9IGNvbmZpZy5tb2RlbAogICAgYWRhcHRlcl9kaXIgPSBjaGVja3BvaW50X2RpciAvICJtb2RlbCIKCiAgICAjIOmHj+WMlumFjee9ru+8iDQtYml0IOmcgOimgSBDVURB77yJCiAgICBibmJfY29uZmlnID0gTm9uZQogICAgaWYgbWMubG9hZF9pbl80Yml0IGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIGJuYl9jb25maWcgPSBCaXRzQW5kQnl0ZXNDb25maWcoCiAgICAgICAgICAgIGxvYWRfaW5fNGJpdD1UcnVlLAogICAgICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPW1jLmJuYl80Yml0X3F1YW50X3R5cGUsCiAgICAgICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudD1tYy5ibmJfNGJpdF91c2VfZG91YmxlX3F1YW50LAogICAgICAgICkKICAgIGVsaWYgbWMubG9hZF9pbl80Yml0OgogICAgICAgIExPR0dFUi53YXJuaW5nKCLmnKrmo4DmtYvliLAgQ1VEQe+8jOi3s+i/hyA0LWJpdCDph4/ljJYiKQoKICAgICMg5Yqg6L296YWN572u5bm25aSE55CGIFZMTSBudW1fbGFiZWxzIOS8oOaSrQogICAgbW9kZWxfY29uZmlnID0gQXV0b0NvbmZpZy5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgbWMubW9kZWxfbmFtZSwKICAgICAgICBudW1fbGFiZWxzPU5VTV9MQUJFTFMsCiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICBjYWNoZV9kaXI9bWMuY2FjaGVfZGlyLAogICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9bWMubG9jYWxfZmlsZXNfb25seSwKICAgICkKICAgIGlmIGhhc2F0dHIobW9kZWxfY29uZmlnLCAidGV4dF9jb25maWciKToKICAgICAgICBtb2RlbF9jb25maWcudGV4dF9jb25maWcubnVtX2xhYmVscyA9IE5VTV9MQUJFTFMKCiAgICAjIOWvuem9kCBwYWRfdG9rZW5faWTvvIjliIbnsbvmqKHlnovpnIDopoHvvIkKICAgIGlmIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWxfY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsX2NvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgICAgIG1vZGVsX2NvbmZpZy50ZXh0X2NvbmZpZy5wYWRfdG9rZW5faWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCgogICAgIyDliqDovb3ln7rluqfmqKHlnosKICAgIGR0eXBlID0gdG9yY2guZmxvYXQxNiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgdG9yY2guZmxvYXQzMgogICAgYmFzZV9tb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1jLm1vZGVsX25hbWUsCiAgICAgICAgY29uZmlnPW1vZGVsX2NvbmZpZywKICAgICAgICBxdWFudGl6YXRpb25fY29uZmlnPWJuYl9jb25maWcsCiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICBjYWNoZV9kaXI9bWMuY2FjaGVfZGlyLAogICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9bWMubG9jYWxfZmlsZXNfb25seSwKICAgICAgICB0b3JjaF9kdHlwZT1kdHlwZSwKICAgICkKCiAgICAjIOWKoOi9vSBMb1JBIGFkYXB0ZXIKICAgIGlmIG1jLnVzZV9sb3JhIGFuZCBhZGFwdGVyX2Rpci5leGlzdHMoKToKICAgICAgICBtb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZV9tb2RlbCwgc3RyKGFkYXB0ZXJfZGlyKSkKICAgICAgICBMT0dHRVIuaW5mbygi5bey5Yqg6L29IExvUkEgYWRhcHRlcjogJXMiLCBhZGFwdGVyX2RpcikKICAgIGVsc2U6CiAgICAgICAgbW9kZWwgPSBiYXNlX21vZGVsCiAgICAgICAgTE9HR0VSLmluZm8oIuacquS9v+eUqCBMb1JBIGFkYXB0ZXLvvIzliqDovb3lrozmlbTmqKHlnosiKQoKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Li75Ye95pWwCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBzZXR1cF9sb2dnaW5nKCkKICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkKICAgIGNoZWNrcG9pbnRfZGlyID0gUGF0aChhcmdzLmNoZWNrcG9pbnRfZGlyKQogICAgc3RhcnRlZF9hdCA9IHRpbWUucGVyZl9jb3VudGVyKCkKCiAgICAjIC0tLS0gMS4g5Yqg6L296YWN572uIC0tLS0KICAgIGNvbmZpZ19wYXRoID0gY2hlY2twb2ludF9kaXIgLyAiY29uZmlnLnlhbWwiCiAgICBpZiBub3QgY29uZmlnX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYi57y65bCR6YWN572u5paH5Lu2OiB7Y29uZmlnX3BhdGh9IikKICAgIGNvbmZpZyA9IGxvYWRfY29uZmlnKGNvbmZpZ19wYXRoKQoKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgTE9HR0VSLmluZm8oIumihOa1i+WQr+WKqCIpCiAgICBMT0dHRVIuaW5mbygiY2hlY2twb2ludDogJXMiLCBjaGVja3BvaW50X2RpcikKICAgIExPR0dFUi5pbmZvKCLorr7lpIc6ICVzIiwgZGVzY3JpYmVfZGV2aWNlKGRldmljZSkpCgogICAgIyAtLS0tIDIuIOWKoOi9vSB0b2tlbml6ZXIgLS0tLQogICAgdG9rZW5pemVyX2RpciA9IGNoZWNrcG9pbnRfZGlyIC8gInRva2VuaXplciIKICAgIGlmIG5vdCB0b2tlbml6ZXJfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIue8uuWwkSB0b2tlbml6ZXIg55uu5b2VOiB7dG9rZW5pemVyX2Rpcn0iKQogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgc3RyKHRva2VuaXplcl9kaXIpLCB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgKQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCiAgICAjIC0tLS0gMy4g5Yqg6L295qih5Z6LIC0tLS0KICAgIG1vZGVsID0gbG9hZF9pbmZlcmVuY2VfbW9kZWwoY2hlY2twb2ludF9kaXIsIGNvbmZpZywgdG9rZW5pemVyKQoKICAgICMgLS0tLSA0LiDliqDovb3mtYvor5XmlbDmja4gLS0tLQogICAgdGVzdF9jc3YgPSBQYXRoKGFyZ3MuZGF0YV9kaXIpIC8gY29uZmlnLmRhdGEudGVzdF9wYXRoCiAgICBMT0dHRVIuaW5mbygi5Yqg6L295rWL6K+V5pWw5o2uOiAlcyIsIHRlc3RfY3N2KQogICAgdGVzdF9kZiA9IGxvYWRfYW5kX3ByZXByb2Nlc3MoCiAgICAgICAgc3RyKHRlc3RfY3N2KSwKICAgICAgICBtYXhfY2hhcnM9Y29uZmlnLmRhdGEudGV4dF9tYXhfY2hhcnMsCiAgICAgICAgaXNfdHJhaW49RmFsc2UsCiAgICApCiAgICB0ZXN0X2lkcyA9IHRlc3RfZGZbImlkIl0udG9saXN0KCkKCiAgICB0ZXN0X2RhdGFzZXQgPSBidWlsZF9kYXRhc2V0KAogICAgICAgIHRlc3RfZGYsIHRva2VuaXplciwgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsIGlzX3RyYWluPUZhbHNlLAogICAgKQogICAgTE9HR0VSLmluZm8oIuW+hemihOa1i+agt+acrDogJXMiLCBsZW4odGVzdF9kYXRhc2V0KSkKCiAgICAjIC0tLS0gNS4g5o6o55CGIC0tLS0KICAgIGRhdGFfY29sbGF0b3IgPSBEYXRhQ29sbGF0b3JXaXRoUGFkZGluZyh0b2tlbml6ZXI9dG9rZW5pemVyLCBwYWRkaW5nPVRydWUpCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIHRlc3RfZGF0YXNldCwKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBzaHVmZmxlPUZhbHNlLAogICAgICAgIGNvbGxhdGVfZm49ZGF0YV9jb2xsYXRvciwKICAgICkKCiAgICBhbGxfcHJvYnMgPSBydW5faW5mZXJlbmNlKGxvYWRlciwgbW9kZWwsIGRldmljZSkKCiAgICBpZiBjb25maWcuZGF0YS5pbmNsdWRlX3N3YXBfdHRhOgogICAgICAgIHN3YXBfZGF0YXNldCA9IGJ1aWxkX2RhdGFzZXQoCiAgICAgICAgICAgIHRlc3RfZGYsCiAgICAgICAgICAgIHRva2VuaXplciwKICAgICAgICAgICAgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsCiAgICAgICAgICAgIGlzX3RyYWluPUZhbHNlLAogICAgICAgICAgICBzd2FwX3BhaXJzPVRydWUsCiAgICAgICAgKQogICAgICAgIHN3YXBfbG9hZGVyID0gRGF0YUxvYWRlcigKICAgICAgICAgICAgc3dhcF9kYXRhc2V0LAogICAgICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgY29sbGF0ZV9mbj1kYXRhX2NvbGxhdG9yLAogICAgICAgICkKICAgICAgICBMT0dHRVIuaW5mbygi5ZCv55SoIHN3YXAgVFRB77yM5YaN6L+Q6KGM5LiA5qyh5a+556ew6aKE5rWLIikKICAgICAgICBzd2FwX3Byb2JzID0gcnVuX2luZmVyZW5jZShzd2FwX2xvYWRlciwgbW9kZWwsIGRldmljZSkKICAgICAgICBzd2FwX3Byb2JzID0gc3dhcF9wcm9ic1s6LCBbMSwgMCwgMl1dCiAgICAgICAgYWxsX3Byb2JzID0gKGFsbF9wcm9icyArIHN3YXBfcHJvYnMpIC8gMi4wCgogICAgIyAtLS0tIDYuIOWGmeWHuiBzdWJtaXNzaW9uLmNzdiAtLS0tCiAgICByb3dzID0gW10KICAgIGZvciBzYW1wbGVfaWQsIHByb2IgaW4gemlwKHRlc3RfaWRzLCBhbGxfcHJvYnMsIHN0cmljdD1UcnVlKToKICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICJpZCI6IHNhbXBsZV9pZCwKICAgICAgICAgICAgSURfVE9fTEFCRUxbMF06IHByb2JbMF0sCiAgICAgICAgICAgIElEX1RPX0xBQkVMWzFdOiBwcm9iWzFdLAogICAgICAgICAgICBJRF9UT19MQUJFTFsyXTogcHJvYlsyXSwKICAgICAgICB9KQoKICAgIG91dHB1dF9wYXRoID0gKAogICAgICAgIFBhdGgoYXJncy5vdXRwdXRfcGF0aCkKICAgICAgICBpZiBhcmdzLm91dHB1dF9wYXRoCiAgICAgICAgZWxzZSBjaGVja3BvaW50X2RpciAvICJzdWJtaXNzaW9uLmNzdiIKICAgICkKICAgIHBkLkRhdGFGcmFtZShyb3dzKS50b19jc3Yob3V0cHV0X3BhdGgsIGluZGV4PUZhbHNlKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIumihOa1i+WujOaIkDogJXMg6KGMLCDogJfml7Y9JXMiLAogICAgICAgIGxlbihyb3dzKSwgZm9ybWF0X3NlY29uZHModGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWRfYXQpLAogICAgKQogICAgTE9HR0VSLmluZm8oInN1Ym1pc3Npb24g5bey5L+d5a2Y5YiwOiAlcyIsIG91dHB1dF9wYXRoKQogICAgcHJpbnQoZiJzYXZlZCBzdWJtaXNzaW9uIHRvIHtvdXRwdXRfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
}

for _name, _b64 in _FILES.items():
    (PKG_DIR / _name).write_text(
        base64.b64decode(_b64).decode('utf-8'),
        encoding='utf-8',
    )
    print(f"  写入 {_name}")

import sys
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))

print("\narena_ranker 已写入:", PKG_DIR)
print("sys.path 已更新")

In [ ]:
import arena_ranker
print('导入成功:', arena_ranker.__file__)

## 3. 配置

设置竞赛数据路径和训练参数。

⚠️ **请根据你的实际竞赛修改 `COMPETITION_SLUG`。**

双 T4 会自动走分布式双卡训练；P100 保持单卡训练。
T4 不支持 bf16，因此默认保持 `BF16 = False`。
当前 Kaggle 环境下，多进程双卡训练默认关闭 `LOAD_IN_4BIT`，避免 4-bit 模型加载卡住。
若 notebook 当前可见两张 GPU 但你强制改成单进程，会自动只暴露 `cuda:0`，避免 Trainer 误退回 DataParallel。

In [ ]:
import torch

# ============================================================
# 🔧 根据你的情况修改以下参数
# ============================================================
COMPETITION_SLUG = "llm-classification-finetuning"   # ← 改成你的竞赛 slug
MODEL_NAME       = "Qwen/Qwen3-0.6B"                # 基座模型
EPOCHS           = 3
BATCH_SIZE       = 2        # per-device batch size
GRAD_ACCUM_STEPS = 4        # T4 x2 时推荐 4；P100 单卡可改回 8
MAX_LENGTH       = 1024     # T4/P100 16GB 推荐 1024
LEARNING_RATE    = 2e-4
USE_LORA         = True
LOAD_IN_4BIT     = True     # 单卡默认开启；双卡会在运行时自动关闭
FP16             = True     # T4 Tensor Cores 建议开启
BF16             = False    # T4 不支持 bf16，请保持关闭
DDP_FIND_UNUSED_PARAMETERS = False
LOCAL_FILES_ONLY = False
# ============================================================

VISIBLE_GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
NUM_PROCESSES = VISIBLE_GPU_COUNT if VISIBLE_GPU_COUNT > 0 else 1
DATA_DIR    = f"/kaggle/input/{COMPETITION_SLUG}"
OUTPUT_DIR  = "/kaggle/working/artifacts/default"

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
effective_batch = BATCH_SIZE * GRAD_ACCUM_STEPS * max(NUM_PROCESSES, 1)
print(f"设备: {device_name} ({vram_gb:.1f} GB)")
print(f"可见 GPU 数量: {VISIBLE_GPU_COUNT}")
print(f"训练进程数: {NUM_PROCESSES}")
print(f"有效 batch size: {effective_batch}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")


In [ ]:
import os

data_files = os.listdir(DATA_DIR)
print("竞赛数据文件:", data_files)
assert "train.csv" in data_files, f"找不到 train.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
assert "test.csv"  in data_files, f"找不到 test.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
print("✅ 数据验证通过")


## 4. 预下载模型（推荐）

为了避免双卡训练时两个进程同时从 HuggingFace 远程拉取同一个模型，
这里会先把远程模型下载到 `/kaggle/working/hf_models/`，随后训练阶段改为只读本地目录。

> 同时会设置 `HF_HUB_DISABLE_XET=1`，优先使用标准 HTTP 下载链路。

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ["HF_HUB_DISABLE_XET"] = "1"

MODEL_RUNTIME_PATH = MODEL_NAME
if "/" in MODEL_NAME and not MODEL_NAME.startswith("/") and not LOCAL_FILES_ONLY:
    from huggingface_hub import snapshot_download

    local_model_dir = Path("/kaggle/working/hf_models") / MODEL_NAME.replace("/", "--")
    local_model_dir.parent.mkdir(parents=True, exist_ok=True)
    MODEL_RUNTIME_PATH = snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(local_model_dir),
        resume_download=True,
    )
    LOCAL_FILES_ONLY = True

print(f"训练时模型路径: {MODEL_RUNTIME_PATH}")
print(f"LOCAL_FILES_ONLY: {LOCAL_FILES_ONLY}")


## 5. 训练

使用 QLoRA 微调 `Qwen3-0.6B`，通过 HuggingFace Trainer 训练。

| 参数 | T4 x2 | P100 x1 | 8GB 显存 | 说明 |
| --- | --- | --- | --- | --- |
| `BATCH_SIZE` | 2 | 2 | 1 | per-device |
| `GRAD_ACCUM_STEPS` | 4 | 8 | 16 | 有效 batch = `batch × grad_accum × GPU数` |
| `MAX_LENGTH` | 1024 | 1024 | 512 | 输入序列最大 token 数 |
| `EPOCHS` | 3 | 3 | 3 | 训练轮数 |
| `LEARNING_RATE` | 2e-4 | 2e-4 | 2e-4 | AdamW 学习率 |
| `LOAD_IN_4BIT` | True | True | True | 4-bit NF4 量化 (QLoRA) |
| `FP16` | True | True | True | 启用混合精度 |
| `BF16` | False | False | False | T4 / P100 都不要开启 |

> **双卡训练说明**: notebook 会自动检测 GPU 数量。
> 当检测到 `T4 x2` 时，会使用 `torch.distributed.run --nproc_per_node=2` 启动双卡训练。
> 若想保持与单卡默认配置接近的有效 batch，推荐 `BATCH_SIZE=2, GRAD_ACCUM_STEPS=4`。

In [ ]:
import os
import shlex
import subprocess
import sys

load_in_4bit_runtime = LOAD_IN_4BIT
if NUM_PROCESSES > 1 and load_in_4bit_runtime:
    load_in_4bit_runtime = False
    print("检测到双卡训练: 已自动关闭 LOAD_IN_4BIT，改用 LoRA + FP16 + DDP")

train_command = [sys.executable]
if NUM_PROCESSES > 1:
    train_command.extend([
        "-m", "torch.distributed.run", "--standalone",
        "--nproc_per_node", str(NUM_PROCESSES),
        "-m", "arena_ranker.train",
    ])
else:
    train_command.extend(["-m", "arena_ranker.train"])

train_command.extend([
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--model-name", MODEL_RUNTIME_PATH,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--max-length", str(MAX_LENGTH),
    "--learning-rate", str(LEARNING_RATE),
])

if not USE_LORA:
    train_command.append("--disable-lora")
if not load_in_4bit_runtime:
    train_command.append("--no-4bit")
if FP16:
    train_command.append("--fp16")
else:
    train_command.append("--no-fp16")
if BF16:
    train_command.append("--bf16")
else:
    train_command.append("--no-bf16")
if DDP_FIND_UNUSED_PARAMETERS:
    train_command.append("--ddp-find-unused-parameters")
else:
    train_command.append("--no-ddp-find-unused-parameters")
if LOCAL_FILES_ONLY:
    train_command.append("--local-files-only")

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working"
env["HF_HUB_DISABLE_XET"] = "1"
env.setdefault("HF_HOME", "/kaggle/working/hf_cache")
if NUM_PROCESSES == 1 and VISIBLE_GPU_COUNT > 1:
    env["CUDA_VISIBLE_DEVICES"] = "0"
    print("单进程训练: 已设置 CUDA_VISIBLE_DEVICES=0，避免 Trainer 使用 DataParallel")
print("训练命令:", " ".join(shlex.quote(part) for part in train_command))
subprocess.run(train_command, env=env, check=True)


## 6. 推理与生成提交文件

加载训练好的 QLoRA adapter + 分类头，对 `test.csv` 推理生成 `submission.csv`。

In [ ]:
import importlib, sys

SUBMISSION_PATH = "/kaggle/working/submission.csv"

sys.argv = [
    "arena-predict",
    "--checkpoint-dir", OUTPUT_DIR,
    "--data-dir",       DATA_DIR,
    "--output-path",    SUBMISSION_PATH,
    "--batch-size",     "4",
]

from arena_ranker.predict import main as predict_main
predict_main()


## 7. 检查提交文件

In [ ]:
import pandas as pd

sub = pd.read_csv(SUBMISSION_PATH)
print(f"行数: {len(sub)}")
print(f"列名: {list(sub.columns)}")
print()
print(sub.head(10))
print()
print("各列概率统计:")
print(sub.describe())
print()

row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(f"概率行和范围: [{row_sums.min():.6f}, {row_sums.max():.6f}]")
print("✅ submission.csv 已生成:", SUBMISSION_PATH)


## 附录 A：离线模式（无需联网）

如果 notebook 不能联网（例如最终提交时），需要提前将模型上传到 Kaggle。

### 步骤

1. **上传模型到 Kaggle**
   - 在本地下载好 `Qwen/Qwen3-0.6B` 的完整文件
   - 前往 [kaggle.com/models](https://kaggle.com/models) → New Model
   - 上传模型文件夹（包含 config.json, model.safetensors 等）
   - 或者使用 Kaggle Datasets 上传也可以

2. **在 notebook 中添加模型数据集**
   - 右侧 Add Input → 搜索你上传的模型

3. **修改配置**
   ```python
   MODEL_NAME = "/kaggle/input/<model-dataset-slug>"  # 改为本地路径
   ```

4. **训练时加上 `--local-files-only`**
   ```python
   sys.argv.append("--local-files-only")
   ```


## 附录 B：将代码上传为 Kaggle Dataset

如果不想在 notebook 里内联代码，可以把仓库上传为 Kaggle Dataset：

1. 打包源码：
   ```bash
   zip -r arena-ranker-code.zip src/ pyproject.toml README.md
   ```

2. 上传到 Kaggle Datasets

3. 在 notebook 中安装：
   ```python
   !pip install /kaggle/input/arena-ranker-code/
   ```

4. 使用 CLI 命令：
   ```python
   !arena-train --data-dir /kaggle/input/<slug>/ --output-dir /kaggle/working/artifacts/default
   !arena-predict --checkpoint-dir /kaggle/working/artifacts/default --data-dir /kaggle/input/<slug>/ --output-path /kaggle/working/submission.csv
   ```


## 附录 C：显存不足时的参数调整

如果遇到 OOM，优先按以下顺序调整：

1. 降低 `MAX_LENGTH`（如 512）
2. 降低 `BATCH_SIZE` 到 1
3. 提高 `GRAD_ACCUM_STEPS`
4. 确保 `LOAD_IN_4BIT = True`

```python
# 8GB 显存推荐参数
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
MAX_LENGTH   = 512
LOAD_IN_4BIT = True
```